In [45]:
# Load necessary libraries

# If packages are missing, install them once and re-run this cell.
required_pkgs <- c("tidyverse", "lubridate", "tsibble", "fable", "feasts", "fabletools", "readr")
missing_pkgs <- required_pkgs[!vapply(required_pkgs, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_pkgs) > 0) {
  install.packages(missing_pkgs)
}

invisible(lapply(required_pkgs, library, character.only = TRUE))

# Larger inline plots in Jupyter/VS Code (IRkernel)
options(repr.plot.width = 12, repr.plot.height = 6, repr.plot.res = 150)

In [46]:
# Load the cleaned data
ilinet <- read_rds("fluview_clean/ilinet_clean.rds")
# nrevss <- read_rds("fluview_clean/nrevss_clean.rds")  # optional

glimpse(ilinet)

Rows: 1,482
Columns: 16
$ region_type            <chr> "National", "National", "National", "National",…
$ region                 <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA,…
$ year                   <int> 1997, 1997, 1997, 1997, 1997, 1997, 1997, 1997,…
$ week                   <int> 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51,…
$ percent_weighted_ili   <dbl> 1.10148, 1.20007, 1.37876, 1.19920, 1.65618, 1.…
$ percent_unweighted_ili <dbl> 1.216860, 1.280640, 1.239060, 1.144730, 1.26112…
$ age_0_4                <int> 179, 199, 228, 188, 217, 178, 294, 288, 268, 29…
$ age_25_49              <int> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA,…
$ age_25_64              <int> 157, 151, 153, 193, 162, 148, 240, 293, 206, 28…
$ age_5_24               <int> 205, 242, 266, 236, 280, 281, 328, 456, 343, 41…
$ age_50_64              <int> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA,…
$ age_65                 <int> 29, 23, 34, 36, 41, 48, 70, 63, 69, 102, 81, 59…
$ ilitotal      

In [50]:
ilinet2 <- ilinet
ilinet2$region_type <- stringr::str_trim(ilinet2$region_type)
ilinet2$region <- stringr::str_trim(ilinet2$region)

# Keep just the National weekly series
y_df <- ilinet2[ilinet2$region_type == "National", c("week_start", "percent_weighted_ili")]
y_df <- y_df[order(y_df$week_start), ]

# tsibble requires each index value to appear once; collapse duplicates if present
dup_weeks <- dplyr::count(y_df, week_start)
dup_weeks <- dplyr::filter(dup_weeks, n > 1)
if (nrow(dup_weeks) > 0) {
  message("Found duplicated week_start rows; averaging percent_weighted_ili within each week.")
  print(dup_weeks)
}

y_df <- dplyr::group_by(y_df, week_start)
y_df <- dplyr::summarise(
  y_df,
  percent_weighted_ili = mean(percent_weighted_ili, na.rm = TRUE),
  .groups = "drop"
 )
y_df$percent_weighted_ili[is.nan(y_df$percent_weighted_ili)] <- NA_real_
y_df <- y_df[order(y_df$week_start), ]

# Make the index an explicit weekly calendar index (enables seasonality with period 52)
y_df$week_start <- tsibble::yearweek(y_df$week_start)

y_tbl <- tsibble::as_tsibble(y_df, index = week_start)

# Ensure a complete weekly grid (adds missing weeks if any)
y_tbl <- tsibble::fill_gaps(y_tbl, .full = TRUE)

# Some models (e.g., STL decomposition) require no missing values.
# Fill gaps by linear interpolation (and carry endpoints) to make a complete weekly series.
if (any(is.na(y_tbl$percent_weighted_ili))) {
  x <- as.numeric(y_tbl$week_start)
  y <- y_tbl$percent_weighted_ili
  y_tbl$percent_weighted_ili <- stats::approx(
    x = x[!is.na(y)],
    y = y[!is.na(y)],
    xout = x,
    method = "linear",
    rule = 2
  )$y
}

dplyr::summarise(
  y_tbl,
  n = dplyr::n(),
  missing_y = sum(is.na(percent_weighted_ili))
 )

Found duplicated week_start rows; averaging percent_weighted_ili within each week.



# A tibble: 5 × 2
  week_start     n
  <date>     <int>
1 1997-12-29     2
2 2003-12-29     2
3 2008-12-29     2
4 2014-12-29     2
5 2025-12-29     2


week_start,n,missing_y
<week>,<int>,<int>
1997 W40,1,0
1997 W41,1,0
1997 W42,1,0
1997 W43,1,0
1997 W44,1,0
1997 W45,1,0
1997 W46,1,0
1997 W47,1,0
1997 W48,1,0


In [51]:
# Split into training and test sets
h <- 104  # forecast horizon (2 years of weekly data)
n_total <- nrow(y_tbl)

train <- y_tbl[1:(n_total - h), ]
test  <- y_tbl[(n_total - h + 1):n_total, ]

range(train$week_start)
range(test$week_start)

<yearweek[2]>
[1] "1997 W40" "2024 W07"
# Week starts on: Monday

<yearweek[2]>
[1] "2024 W08" "2026 W07"
# Week starts on: Monday

In [52]:
fit_baselines <- fabletools::model(
  train,
  mean   = fable::MEAN(percent_weighted_ili), # mean baseline: constant forecast = training mean
  # Weekly data: 1 year = 52 weeks
  snaive = fable::SNAIVE(percent_weighted_ili ~ lag(52)), # seasonal naïve: “this week will look like the same week last year”
  # ETS() cannot directly handle seasonal periods > 24, so we model weekly seasonality via STL + ETS.
  ets    = fabletools::decomposition_model(
    feasts::STL(percent_weighted_ili ~ season(period = 52)),
    fable::ETS(season_adjust ~ error("A") + trend("Ad"))
  ),
  arima  = fable::ARIMA(percent_weighted_ili)
 )

fit_baselines

fc_baselines <- fabletools::forecast(fit_baselines, h = h)

# Note: these models assume an unbounded (approximately Gaussian) error distribution on this scale,
# so prediction intervals can extend below 0 even if the true series can't.
autoplot(train, percent_weighted_ili) +
  autolayer(fc_baselines, alpha = 0.7) +
  autolayer(test, percent_weighted_ili, color = "black") +
  labs(title = "Baseline forecasts vs actual", x = "Week", y = "ILI (%)")

acc_baselines <- fabletools::accuracy(fc_baselines, test)
acc_baselines <- dplyr::select(acc_baselines, .model, RMSE)
acc_baselines

# Model summaries
# `report()` is only supported for individual models (not the whole mdl_df at once).
# Printing as plain text avoids occasional rich-display issues in VS Code/Jupyter.
cat(paste(capture.output(fabletools::report(dplyr::select(fit_baselines, mean))), collapse = "\n"), "\n\n")
cat(paste(capture.output(fabletools::report(dplyr::select(fit_baselines, snaive))), collapse = "\n"), "\n\n")
cat(paste(capture.output(fabletools::report(dplyr::select(fit_baselines, ets))), collapse = "\n"), "\n\n")
cat(paste(capture.output(fabletools::report(dplyr::select(fit_baselines, arima))), collapse = "\n"), "\n")

Warning message:
"Seasonal periods (`period`) of length greather than 24 are not supported by
ETS. Seasonality will be ignored."


: 